# RakshaVaani - voice clone detector

frozen XLS-R backbone (facebook/wav2vec2-xls-r-300m) + a small classifier on top.
not fine-tuning the whole thing bc that'd take forever on a laptop, this is way faster
and still gets decent numbers.

setup before running (terminal, inside the project folder):

```
python3 -m venv venv
source venv/bin/activate
pip install torch torchaudio transformers librosa soundfile scikit-learn kagglehub pandas numpy matplotlib jupyter
jupyter notebook
```

kaggle.json needs to be in ~/.kaggle/ for the download to work (get it from kaggle account settings)


In [31]:
import os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torchaudio

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(device)

PROJECT_DIR = os.path.expanduser("~/Desktop/vaaniraksha")
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)


mps


download dataset from kaggle. goes to local disk not drive so no quota issues

In [32]:
import kagglehub

KAGGLE_DATASET = "buleboy/lazip"

t0 = time.time()
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
print(time.time()-t0, "sec")
print(dataset_path)


1.62363600730896 sec
/Users/ayushpandey/.cache/kagglehub/datasets/buleboy/lazip/versions/1


In [33]:
# just checking what we actually got, folder structure isnt always what you expect
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:5]:
            print(f"{indent}  {f}")


1/
  LA/
    README.LA.txt
    ASVspoof2019_LA_asv_protocols/
      ASVspoof2019.LA.asv.eval.gi.trl.txt
      ASVspoof2019.LA.asv.eval.female.trl.txt
      ASVspoof2019.LA.asv.dev.male.trl.txt
      ASVspoof2019.LA.asv.dev.male.trn.txt
      ASVspoof2019.LA.asv.eval.female.trn.txt
    ASVspoof2019_LA_train/
      LICENSE.txt
      flac/
    ASVspoof2019_LA_dev/
      LICENSE.txt
      flac/
    ASVspoof2019_LA_cm_protocols/
      ASVspoof2019.LA.cm.dev.trl.txt
      ASVspoof2019.LA.cm.eval.trl.txt
      ASVspoof2019.LA.cm.train.trn.txt
    ASVspoof2019_LA_asv_scores/
      ASVspoof2019.LA.asv.eval.gi.trl.scores.txt
      ASVspoof2019.LA.asv.dev.gi.trl.scores.txt
    ASVspoof2019_LA_eval/
      LICENSE.txt
      flac/


this is ASVspoof2019 LA. labels come from the protocol txt file, not folder names

In [34]:
LA_ROOT = os.path.join(dataset_path, "LA")
protocol_path = os.path.join(LA_ROOT, "ASVspoof2019_LA_cm_protocols", "ASVspoof2019.LA.cm.train.trn.txt")
flac_dir = os.path.join(LA_ROOT, "ASVspoof2019_LA_train", "flac")

proto = pd.read_csv(protocol_path, sep=r"\s+", header=None,
                     names=["speaker_id", "filename", "unused", "system_id", "label"])
proto["label"].value_counts()


label
spoof       22800
bonafide     2580
Name: count, dtype: int64

In [35]:
LABEL_MAP = {"bonafide": 0, "spoof": 1}

rows = []
for _, r in proto.iterrows():
    fp = os.path.join(flac_dir, r["filename"] + ".flac")
    if os.path.exists(fp):
        rows.append({"filepath": fp, "label": LABEL_MAP[r["label"]]})

df = pd.DataFrame(rows)
print(len(df), "files found")
df["label"].value_counts()


25380 files found


label
1    22800
0     2580
Name: count, dtype: int64

telephony sim - real scam calls are 8khz, compressed, noisy. not clean studio audio like
most of these datasets. so we degrade the audio before feeding it to the model to make it
robust to what a real phone call actually sounds like

In [36]:
import torchaudio.functional as AF

def simulate_phone_call(waveform, orig_sr, noise_level=0.005):
    target_sr = 8000
    wav_8k = AF.resample(waveform, orig_sr, target_sr)
    # mu law encode/decode = codec compression artifact, similar to what G.711 does on real calls
    wav_mu = AF.mu_law_encoding(wav_8k, quantization_channels=256)
    wav_decoded = AF.mu_law_decoding(wav_mu, quantization_channels=256)
    noise = torch.randn_like(wav_decoded) * noise_level
    wav_noisy = wav_decoded + noise
    wav_final = AF.resample(wav_noisy, target_sr, 16000)  # xls-r wants 16k
    return wav_final


load the backbone, freeze it, just use it for embeddings

In [37]:
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
backbone = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(device)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [38]:
import soundfile as sf

# using soundfile instead of torchaudio.load, torchaudio's new backend wants torchcodec
# which was being annoying to install so just skip it
@torch.no_grad()
def get_embedding(filepath, apply_telephony_sim=False):
    audio, sr = sf.read(filepath, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    waveform = torch.from_numpy(audio).unsqueeze(0)

    if sr != 16000:
        waveform = AF.resample(waveform, sr, 16000)
        sr = 16000
    if apply_telephony_sim:
        waveform = simulate_phone_call(waveform, sr)

    inputs = feature_extractor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
    input_values = inputs.input_values.to(device)
    outputs = backbone(input_values)
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return embedding

# quick sanity check
get_embedding(df.iloc[0]["filepath"]).shape


(1024,)

grabbing a subset to embed, doing the whole dataset would take way too long.
400 per class is a decent tradeoff between speed and having enough data

In [39]:
N_PER_CLASS = 400

df_bal = pd.concat([
    df[df.label == 0].sample(min((df.label == 0).sum(), N_PER_CLASS), random_state=42),
    df[df.label == 1].sample(min((df.label == 1).sum(), N_PER_CLASS), random_state=42),
]).reset_index(drop=True)

print(len(df_bal), df_bal["label"].value_counts().to_dict())

t0 = time.time()
X_clean, X_phone, y = [], [], []
for i, row in df_bal.iterrows():
    try:
        X_clean.append(get_embedding(row["filepath"], apply_telephony_sim=False))
        X_phone.append(get_embedding(row["filepath"], apply_telephony_sim=True))
        y.append(row["label"])
    except Exception as e:
        print("skipped", row["filepath"], e)
    if (i + 1) % 50 == 0:
        print(i+1, "/", len(df_bal), time.time()-t0)

X_clean = np.array(X_clean)
X_phone = np.array(X_phone)
y = np.array(y)
print((time.time()-t0)/60, "min total")


800 {0: 400, 1: 400}
50 / 800 9.929996967315674
100 / 800 19.619181871414185
150 / 800 28.5142822265625
200 / 800 37.398473024368286
250 / 800 46.68889403343201
300 / 800 55.228413105010986
350 / 800 63.40299892425537
400 / 800 71.96083188056946
450 / 800 81.1323471069336
500 / 800 90.44238901138306
550 / 800 100.23777198791504
600 / 800 109.12792992591858
650 / 800 118.6906578540802
700 / 800 127.21830105781555
750 / 800 135.8935329914093
800 / 800 144.6485559940338
2.410985052585602 min total


for saving the trained file

In [40]:
import joblib
joblib.dump(clf, os.path.join(PROJECT_DIR, "rakshavaani_classifier_head.joblib"))

['/Users/ayushpandey/Desktop/vaaniraksha/rakshavaani_classifier_head.joblib']

train on the phone-simulated embeddings since thats the realistic condition

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_curve

Xtr, Xte, ytr, yte = train_test_split(X_phone, y, test_size=0.25, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(Xtr, ytr)

pred = clf.predict(Xte)
print(accuracy_score(yte, pred))


0.85


EER - standard metric for spoof detection. also checking clean vs phone-condition gap

In [42]:
def compute_eer(y_true, scores):
    fpr, tpr, thresholds = roc_curve(y_true, scores)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[idx] + fnr[idx]) / 2 * 100

scores_phone = clf.predict_proba(Xte)[:, 1]
eer_phone = compute_eer(yte, scores_phone)
print("EER phone:", eer_phone)

_, Xte_clean_idx = train_test_split(np.arange(len(y)), test_size=0.25, random_state=42, stratify=y)
scores_clean = clf.predict_proba(X_clean[Xte_clean_idx])[:, 1]
eer_clean = compute_eer(y[Xte_clean_idx], scores_clean)
print("EER clean:", eer_clean)


EER phone: 18.0
EER clean: 21.499999999999996


In [ ]:
import joblib
joblib.dump(clf, os.path.join(PROJECT_DIR, "rakshavaani_classifier_head.joblib"))


inference fn for the demo

In [43]:
def detect_voice_clone(filepath):
    t0 = time.time()
    emb = get_embedding(filepath, apply_telephony_sim=True)
    prob_spoof = clf.predict_proba(emb.reshape(1, -1))[0, 1]
    verdict = "AI CLONED" if prob_spoof > 0.5 else "REAL"
    confidence = prob_spoof if prob_spoof > 0.5 else 1 - prob_spoof
    print(f"{os.path.basename(filepath)} -> {verdict} ({confidence*100:.1f}% conf, {time.time()-t0:.2f}s)")
    return verdict, confidence


In [44]:
demo_files = [
    df_bal[df_bal.label == 0].iloc[0]["filepath"],
    df_bal[df_bal.label == 1].iloc[0]["filepath"],
]
for f in demo_files:
    detect_voice_clone(f)


LA_T_3891170.flac -> REAL (61.7% conf, 0.57s)
LA_T_7196016.flac -> AI CLONED (80.0% conf, 0.07s)
